In [5]:
from langchain_ollama import ChatOllama

local_llm = "llama3.2:3b"
llm = ChatOllama(model=local_llm, temperature=0)
llm_json_mode = ChatOllama(model=local_llm, temperature=0, format="json")

In [6]:
import os, getpass


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("TAVILY_API_KEY")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=3)

In [8]:
import operator
from typing_extensions import TypedDict
from typing import List, Annotated


class GraphState(TypedDict):
    """dictionary that contains information we want to propagate to, and modify in, each graph node."""

    chat_message: str  # User question
    generation: str  # LLM generation
    max_retries: int  # Max number of retries for answer generation
    answers: int  # Number of answers generated
    loop_step: Annotated[int, operator.add]

In [9]:
router_instructions = """You are an expert at routing a user question to either worker1 or worker2.
The worker1 is an LLM containing information on Customer Service any issues related to that must be directed to it.                    
Use the worker2 for information regarding products and things like that which can be only informational not helpful.
Return JSON with single key, datasource, that is 'worker1' worker2' depending on the question.
"""
# You are a grader assessing relevance of a retrieved document to a user question.
# If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
# doc_grader_instructions = """You are a grader assessing relevance of a retrieved document to a user chat message.
# If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.
# Return JSON with single key, binary_score, that is 'yes' or 'no' score to indicate whether the document contains at least some information that is relevant to the question.
# """

# # Here is the retrieved document: \n\n {document} \n\n Here is the user question: \n\n {question}.
# # This carefully and objectively assess whether the document contains at least some information that is relevant to the question.
# # Return JSON with single key, binary_score, that is 'yes' or 'no' score to indicate whether the document contains at least some information that is relevant to the question."""


# doc_grader_prompt = """Here is the retrieved document: \n\n {document} \n\n Here are few of the user chat message \n\n {chat_message}
# Observe this carefully and give a binary score to indicate whether the document contains at least some information that is relevant to the question.
# Return JSON with single key, binary_score, that is 'yes' or 'no' score to indicate whether the document contains at least some information that is relevant to the question.
# """

# rag_prompt = """You are responding as a person to a chat response. Here is the context to use to answer the question:
# {context}
# Think about this context carefully and give an appropriate response to the following set of chat messages: {chat_message}
# Provide reply in a conversational manner using only the context provided. Use one sentence maximum.
# """

In [10]:
import json
from langchain.schema import Document
from langgraph.graph import END
from langchain_core.messages import HumanMessage, SystemMessage

In [11]:
def route_question(state):
    """Route question to web search or RAG"""

    print("---ROUTE QUESTION---")
    route_question = llm_json_mode.invoke(
        [SystemMessage(content=router_instructions)]
        + [HumanMessage(content=state["chat_message"])]
    )
    source = json.loads(route_question.content)["datasource"]
    if source == "quick-response":
        print("---ROUTE QUESTION TO QUICK-RESPONSE---")
        return "worker1"
    elif source == "vectorstore":
        print("---ROUTE QUESTION TO RAG---")
        return "worker2"


def worker1(state):
    generation = llm.invoke([HumanMessage(content=state["chat_message"])])
    return {"generation": generation}


def worker2(state):
    generation = llm.invoke([HumanMessage(content=state["chat_message"])])
    return {"generation": generation}

In [12]:
from langgraph.graph import StateGraph
from IPython.display import Image, display

workflow = StateGraph(GraphState)

workflow.add_node("worker1", worker1)
workflow.add_node("worker2", worker2)

workflow.set_conditional_entry_point(
    route_question, {"worker1": "worker1", "worker2": "worker2"}
)

In [13]:
g = workflow.compile()
display(Image(g.get_graph().draw_mermaid_png()))

ValueError: Node `route_question` is not reachable